# ✦ Business-SK · render GPU (Google Colab)

Use this when your **laptop is off** — from a laptop **or your phone**. This Colab becomes the render GPU for your posts (product cut-outs + AI backdrops, the same models the laptop runs).

**📱 On your phone** (Chrome, signed in to the same Google account)
1. Tap the **▶ play button** on the cell below — that's all. It picks the T4 GPU, installs, connects and starts rendering.
2. Keep this tab open with the screen on while posts render (plug the phone in). Android pauses background tabs — if Colab stops, SK Helper notifies you; tap the notification and press ▶ again.

**💻 On a laptop:** press ▶ on the cell below, or **Runtime → Run all**.

**Connecting:** your saved key in 🔑 **Secrets** (`SK_COLAB_KEY`, made once in Studio → Overview → Render GPUs → **Create key**) is used automatically — on the phone too, because Secrets belong to your Google account. No key? Type a one-time code from the Studio (**Get Colab code**) into `SESSION_CODE` below, or paste it when it asks.

The first start downloads the models (a few minutes). Google decides how much free GPU time you get.


In [ ]:
#@title ▶ Start rendering (tap play — phone or laptop)
SESSION_CODE = ""  #@param {type:"string"}
STOP_AFTER_IDLE_MINUTES = 0  #@param {type:"integer"}
# SESSION_CODE: only if you have no saved key · STOP_AFTER_IDLE_MINUTES: 0 = keep running; 30 = stop after 30 idle minutes (saves free GPU time)
import json, os, subprocess, sys, urllib.error, urllib.request
STUDIO = "https://140-238-247-18.nip.io"

# 1 · GPU
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU here — Runtime → Change runtime type → T4 GPU (phone: ⋮ menu → Runtime), then ▶ again.")
print("✔ GPU:", gpu.stdout.strip())

# 2 · connect to your Studio — saved key (🔑 Secrets → SK_COLAB_KEY) or a one-time code
key = ""
try:
    from google.colab import userdata
    key = (userdata.get("SK_COLAB_KEY") or "").strip()
except Exception:
    key = ""
if key and not SESSION_CODE.strip():
    os.environ["GPU_WORKER_TOKEN"] = key
    print("✔ Connected with your saved Colab key")
else:
    code = SESSION_CODE.strip() or input("Session code from the Studio (e.g. ABCD-2345): ").strip()
    req = urllib.request.Request(STUDIO + "/api/gpu/colab/claim", data=json.dumps({"code": code}).encode(),
                                 headers={"Content-Type": "application/json"})
    try:
        os.environ["GPU_WORKER_TOKEN"] = json.load(urllib.request.urlopen(req, timeout=30))["token"]
    except urllib.error.HTTPError as e:
        raise SystemExit("Code not accepted: " + e.read().decode("utf-8", "ignore")[:200])
    print("✔ Connected — this Colab is your render GPU for the next 12 hours")

# 3 · libraries (same as the laptop, ~1 minute; quiet)
print("… installing the render libraries")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.40.0", "transformers==5.17.0", "accelerate==1.15.0",
                "bitsandbytes==0.50.2", "timm", "kornia", "einops"], check=True)
print("✔ Libraries ready")

# 4 · render until the session ends (keep this tab open)
urllib.request.urlretrieve("https://raw.githubusercontent.com/Skarthik06/business-sk/main/scripts/gpu_worker.py", "gpu_worker.py")
os.environ.update(SK_API=STUDIO, SK_WORKER_KIND="colab", SK_IDLE_EXIT_SECS=str(max(0, int(STOP_AFTER_IDLE_MINUTES)) * 60))
print("✔ Rendering — the Studio shows ‘Colab online’. Keep this tab open.")
!python gpu_worker.py
